# Concentração das emissões e alcance dos grandes emissores

**Pergunta:** o CO₂ da produção brasileira se concentra em poucos setores e as cadeias dos demais setores dependem muito dessas origens emissoras?

Separamos três dimensões: (1) volume próprio emitido, (2) participação desse emissor no CO₂ atribuído a cada demanda final e (3) fornecimento direto de insumos nacionais. Usamos a MIP de 2015 e P, não C nem o vetor de consumo. Os valores são estimativas do modelo com intensidades interpoladas de 2011/2018 e a aproximação monetária adotada no projeto; não são um inventário observado de 2015.

A rede é dirigida: **origem emissora i → atividade da demanda final j**. P já incorpora encadeamentos diretos e indiretos, incluindo exportações. Seus caminhos não devem ser propagados novamente como se fossem etapas diretas de produção. Similaridade de cosseno e clusters comparam perfis, mas não respondem diretamente à pergunta de concentração e alcance.

Execute este notebook da raiz do repositório. Ele verifica as entradas pelo manifesto e grava somente `outputs/concentracao_dependencia/`.

In [1]:
from pathlib import Path
from hashlib import sha256
import platform
import numpy as np
import pandas as pd
import networkx as nx
import plotly.graph_objects as go
from IPython.display import display
from redes.redes import carregar_matriz_emissoes, carregar_setores
from redes import dados, mip, modelo

P = carregar_matriz_emissoes("matriz_emissoes_producao_2015")
setores = carregar_setores("setores_mip_2015").reindex(P.index)
assert setores.notna().all() and P.to_numpy().sum() > 0
P_original = P.copy()
pasta = Path("outputs/concentracao_dependencia")
pasta.mkdir(exist_ok=True)
print("Python", platform.python_version(), "NumPy", np.__version__, "pandas", pd.__version__, "NetworkX", nx.__version__)


SHA-256 verificado: matriz_emissoes_producao_2015.csv
SHA-256 verificado: setores_2015.csv
Python 3.12.10 NumPy 2.2.6 pandas 3.0.0 NetworkX 3.6.1


## 1. Volume, alcance e dependência de emissões

**Emissões próprias da atividade:** $e_i=\sum_j P_{ij}=\gamma_i x_i$. A diagonal permanece neste total. A participação $e_i/\sum e_i$ responde quem emite mais.

**Rede entre atividades distintas:** $W_{ij}=P_{ij}$ para $i\ne j$, com diagonal zero. A fração $\sum_j W_{ij}/e_i$ informa quanto da emissão própria atende à demanda final de outras atividades.

**Participação do emissor na pegada doméstica do destino:**

$$R_{ij}=\frac{W_{ij}}{\sum_k P_{kj}}.$$

O denominador inclui as emissões próprias do destino. Por exemplo, $R_{ij}=0,10$ significa que 10% das emissões brasileiras atribuídas à demanda final de j surgem na atividade i. Isso mede dependência da cadeia em termos de CO₂, não indispensabilidade tecnológica nem impossibilidade de substituição.

Também calculamos $D_{ij}=W_{ij}/\sum_k W_{kj}$, já usado na análise de redes anterior: participação somente nas emissões geradas em **outras** atividades. Os dois denominadores respondem perguntas diferentes. Colunas sem emissões no respectivo universo ficam indefinidas (`NaN`).

Contamos destinos com participação ≥ **1%, 5% e 10%**, excluindo o próprio setor. São limiares descritivos; nenhum deles define por si só uma dependência crítica. A sensibilidade ao limiar acompanha o resultado. O grafo principal usa 5% de R e mantém todos os 67 nós, inclusive isolados.

In [2]:
emissoes = P.sum(axis=1)
pegada_destinos = P.sum(axis=0)
diagonal = pd.Series(np.diag(P.to_numpy()), index=P.index)
valores_w = P.to_numpy(copy=True)
np.fill_diagonal(valores_w, 0)
W = pd.DataFrame(valores_w, index=P.index, columns=P.columns)
R = W.div(pegada_destinos.replace(0, np.nan), axis=1)
D = W.div(W.sum(axis=0).replace(0, np.nan), axis=1)
limiares = [0.01, 0.05, 0.10]

metricas = pd.DataFrame({"descricao": setores, "emissoes_gg": emissoes,
                         "participacao_emissoes": emissoes/emissoes.sum(),
                         "fracao_outros_destinos": W.sum(axis=1)/emissoes.replace(0, np.nan),
                         "destinos_com_ligacao": W.gt(0).sum(axis=1)})
for limiar in limiares:
    sufixo = f"{int(limiar*100)}pct"
    metricas[f"alcance_total_{sufixo}"] = R.ge(limiar).sum(axis=1)
    metricas[f"alcance_externo_{sufixo}"] = D.ge(limiar).sum(axis=1)

G = nx.DiGraph()
G.add_nodes_from(P.index)
nx.set_node_attributes(G, emissoes.to_dict(), "emissoes_gg")
for origem in P.index:
    for destino in P.columns:
        if origem != destino and R.loc[origem, destino] >= 0.05:
            G.add_edge(origem, destino, weight=float(R.loc[origem, destino]),
                       emissoes_gg=float(W.loc[origem, destino]))
np.testing.assert_array_equal(pd.Series(dict(G.out_degree())).reindex(P.index), metricas.alcance_total_5pct)
np.testing.assert_allclose(W.sum(axis=1)+diagonal, emissoes)
validos = pegada_destinos > 0
np.testing.assert_allclose(R.loc[:, validos].sum(axis=0),
                           1-diagonal[validos]/pegada_destinos[validos])
assert nx.number_of_selfloops(G) == 0
ranking = metricas.sort_index().sort_values("emissoes_gg", ascending=False, kind="stable")
densidade_positiva = W.gt(0).to_numpy().sum() / (len(P)*(len(P)-1))
print(f"Rede sem limiar: {densidade_positiva:.2%} das ligações possíveis.")
print(f"Rede R ≥ 5%: {G.number_of_nodes()} nós e {G.number_of_edges()} arestas.")
display(ranking.head(10))


Rede sem limiar: 97.01% das ligações possíveis.
Rede R ≥ 5%: 67 nós e 92 arestas.
                                                            descricao  ...  alcance_externo_10pct
atividade_emissora                                                     ...                       
3500                Energia elétrica, gás natural e outras utilidades  ...                     34
3680                                Água, esgoto e gestão de resíduos  ...                     18
4180                                                       Construção  ...                      0
1100                                            Fabricação de bebidas  ...                      2
1091                Abate e produtos de carne, inclusive os produt...  ...                      1
1093                                      Outros produtos alimentares  ...                      3
4580                                    Comércio por atacado e varejo  ...                      0
8400                Administração pú

## 2. A emissão se concentra em poucos setores?

Calculamos a participação dos 5 e 10 maiores, o menor número de setores que alcança 50% e 80%, o Gini e o HHI. O número efetivo $1/HHI$ resume quantos emissores de igual peso produziriam a mesma concentração. Não adotamos limites de concentração de mercados como teste ambiental. A concentração depende do nível de agregação: aqui são 67 atividades.

In [3]:
participacoes = ranking.participacao_emissoes
acumulado = participacoes.cumsum()
n50 = int(np.searchsorted(ranking.emissoes_gg.cumsum().to_numpy(), 0.5*emissoes.sum()) + 1)
n80 = int(np.searchsorted(ranking.emissoes_gg.cumsum().to_numpy(), 0.8*emissoes.sum()) + 1)
hhi = float(np.square(participacoes).sum())
e = emissoes.to_numpy()
gini = np.abs(e[:, None]-e[None, :]).sum()/(2*len(e)*e.sum())
concentracao = pd.DataFrame([{
    "setores": len(P), "total_mt": e.sum()/1000,
    "top5": participacoes.head(5).sum(), "top10": participacoes.head(10).sum(),
    "n50": n50, "n80": n80, "gini": gini, "hhi": hhi, "emissores_efetivos": 1/hhi,
}])
display(concentracao)


   setores    total_mt      top5  ...      gini       hhi  emissores_efetivos
0       67  683.696041  0.303045  ...  0.503992  0.032857           30.435023

[1 rows x 9 columns]


## 3. Os maiores emissores são relevantes para os demais?

Para o grupo H dos maiores emissores, somamos as origens **sem duplicação**:

$$r_{Hj}=\frac{\sum_{i\in H}P_{ij}}{\sum_i P_{ij}},\qquad j\notin H.$$

Excluímos os próprios membros de H do conjunto de destinos avaliado. Mostramos o grupo dos 5 maiores, dos 10 maiores e o menor grupo que alcança 50% das emissões. A mediana trata os destinos com o mesmo peso. O gráfico de dispersão distingue grandes emissores de emissores com amplo alcance; nenhum escore composto mistura as duas dimensões.

In [4]:
alcance_grupos = []
dependencias_grupos = []
for k in sorted(set([5, 10, n50])):
    grupo = ranking.index[:k]
    outros = P.columns[~P.columns.isin(grupo)]
    fracao = P.loc[grupo, outros].sum(axis=0)/pegada_destinos.loc[outros].replace(0, np.nan)
    for destino, valor in fracao.items():
        dependencias_grupos.append({"top_k": k, "destino": destino, "descricao": setores[destino],
                                   "participacao_grupo": valor,
                                   "emissoes_grupo_gg": P.loc[grupo, destino].sum()})
    for limiar in [0.01, 0.05, 0.10, 0.20, 0.50]:
        alcance_grupos.append({"top_k": k, "participacao_emissoes": participacoes.head(k).sum(),
                               "destinos_fora_grupo": len(outros), "destinos_validos": fracao.notna().sum(),
                               "limiar": limiar, "destinos_acima_limiar": fracao.ge(limiar).sum(),
                               "mediana_participacao": fracao.median(), "minimo_participacao": fracao.min(),
                               "maximo_participacao": fracao.max()})
alcance_grupos = pd.DataFrame(alcance_grupos)
dependencias_grupos = pd.DataFrame(dependencias_grupos)
display(alcance_grupos)


    top_k  participacao_emissoes  ...  minimo_participacao  maximo_participacao
0       5               0.303045  ...                  0.0             0.478006
1       5               0.303045  ...                  0.0             0.478006
2       5               0.303045  ...                  0.0             0.478006
3       5               0.303045  ...                  0.0             0.478006
4       5               0.303045  ...                  0.0             0.478006
5      10               0.482836  ...                  0.0             0.576556
6      10               0.482836  ...                  0.0             0.576556
7      10               0.482836  ...                  0.0             0.576556
8      10               0.482836  ...                  0.0             0.576556
9      10               0.482836  ...                  0.0             0.576556
10     11               0.506849  ...                  0.0             0.583541
11     11               0.506849  ...   

## 4. Conectividade econômica direta: uma verificação distinta

R já contém encadeamentos indiretos. Para olhar as compras diretas nacionais, extraímos **A da Tabela 14 do IBGE**, cujo elemento $A_{ij}$ é o valor de insumos de i por unidade monetária de produção bruta de j. Contamos compradores distintos com $A_{ij}\ge0,01$: ao menos R$ 1 de insumo nacional de i por R$ 100 produzidos em j.

Esse limiar monetário não é comparável ao limiar de 1% das emissões de R. A não inclui os insumos importados. Nenhuma dessas medidas prova que um fornecedor seja insubstituível; isso exigiria hipóteses adicionais de substituição ou cenários de choque.

Conferimos também $e=\gamma x$ com a produção bruta e os coeficientes usados pelo projeto, para manter a interpretação de emissões próprias explícita.

In [5]:
tabelas = dados.carregar_matriz_67("mip_ibge_2015_67")
A = mip.carregar_coeficientes_tecnicos_67(tabelas)
assert A.index.equals(P.index) and A.columns.equals(P.columns)
valores_a = A.to_numpy(copy=True)
np.fill_diagonal(valores_a, 0)
A_sem_diagonal = pd.DataFrame(valores_a, index=A.index, columns=A.columns)
metricas["compradores_diretos_A_1pct"] = A_sem_diagonal.ge(0.01).sum(axis=1)

x = modelo.calcular_producao_bruta_67(tabelas)
coef_2011 = dados.carregar_coeficientes_co2("sanguinet_azzoni_2011")[2011]
coef_2018 = dados.carregar_coeficientes_co2("sanguinet_azzoni_2018")[2018]
gamma = coef_2011 + (4/7)*(coef_2018-coef_2011)
# A correspondência S1…S67 → códigos da MIP segue a ordem da fonte usada no notebook principal.
gamma.index = x.index
np.testing.assert_allclose(emissoes, (gamma*x).reindex(P.index), rtol=1e-10, atol=1e-8)
ranking = metricas.loc[ranking.index]
pd.testing.assert_frame_equal(P, P_original)
display(ranking.head(10))


SHA-256 verificado: Matriz_de_Insumo_Produto_2015_Nivel_67.xls.original
SHA-256 verificado: coeficientes_co2_sanguinet_azzoni_2011.csv
SHA-256 verificado: coeficientes_co2_sanguinet_azzoni_2018.csv
                                                            descricao  ...  compradores_diretos_A_1pct
atividade_emissora                                                     ...                            
3500                Energia elétrica, gás natural e outras utilidades  ...                          24
3680                                Água, esgoto e gestão de resíduos  ...                           6
4180                                                       Construção  ...                          12
1100                                            Fabricação de bebidas  ...                           2
1091                Abate e produtos de carne, inclusive os produt...  ...                           6
1093                                      Outros produtos alimentares  ...       

## 5. Figuras que respondem à pergunta

1. Curva acumulada: quantos setores concentram quanto das emissões.
2. Dispersão: emissão própria versus número de outras atividades cuja pegada brasileira contém pelo menos 5% daquele emissor. A cor informa a fração das emissões do emissor atribuída a outras demandas finais.
3. Matriz da rede R para os dez maiores emissores, em porcentagem da pegada total do destino. A diagonal fica mascarada apenas na figura. Os destinos são ordenados pela participação conjunta dos cinco maiores emissores, com os setores do próprio grupo ao final.

As tabelas preservam os resultados para 1%, 5% e 10% e para o denominador que considera somente outras origens emissoras.

In [6]:
fig_concentracao = go.Figure()
fig_concentracao.add_trace(go.Scatter(x=np.arange(len(P)+1), y=np.r_[0, acumulado.to_numpy()]*100,
                                     mode="lines+markers", name="Maiores emissores primeiro"))
fig_concentracao.add_trace(go.Scatter(x=[0,len(P)], y=[0,100], mode="lines",
                                     line=dict(dash="dash", color="gray"), name="Emissões iguais"))
fig_concentracao.update_layout(title="Concentração do CO₂ da produção brasileira", xaxis_title="Número de setores",
                               yaxis_title="Participação acumulada nas emissões (%)", height=480)

fig_alcance = go.Figure(go.Scatter(
    x=ranking.participacao_emissoes*100, y=ranking.alcance_total_5pct,
    mode="markers+text", text=[codigo if codigo in ranking.index[:10] else "" for codigo in ranking.index],
    textposition="top center", marker=dict(size=11, color=ranking.fracao_outros_destinos,
                                             colorscale="Viridis", cmin=0, cmax=1,
                                             colorbar=dict(title="Fração para<br>outros destinos")),
    customdata=np.column_stack([ranking.index, ranking.descricao, ranking.emissoes_gg/1000,
                               ranking.alcance_total_1pct, ranking.alcance_total_10pct]),
    hovertemplate="%{customdata[0]} — %{customdata[1]}<br>Emissões próprias: %{customdata[2]:.2f} Mt"
                  "<br>Participação: %{x:.2f}%<br>Destinos com ≥5%: %{y}"
                  "<br>Com ≥1%: %{customdata[3]} | Com ≥10%: %{customdata[4]}<extra></extra>",
))
fig_alcance.update_layout(title="Emitir muito e ter amplo alcance são dimensões distintas",
                          xaxis_title="Participação nas emissões próprias (%)",
                          yaxis_title="Outros destinos com ≥5% de participação do emissor",
                          height=560)

top5 = ranking.index[:5]
outros = P.columns[~P.columns.isin(top5)]
ordem = (P.loc[top5, outros].sum(axis=0)/pegada_destinos.loc[outros]).sort_values(ascending=False, kind="stable").index.tolist() + top5.tolist()
top10 = ranking.index[:10]
mapa = R.loc[top10, ordem].copy()
for codigo in top10:
    mapa.loc[codigo, codigo] = np.nan
fig_dependencia = go.Figure(go.Heatmap(
    z=mapa.to_numpy()*100, x=ordem, y=top10.tolist(), colorscale="YlOrRd", zmin=0, zmax=100,
    customdata=np.array([[[setores[i], setores[j], float(P.loc[i,j])] for j in ordem] for i in top10], dtype=object),
    colorbar=dict(title="% da pegada<br>do destino"), hoverongaps=False,
    hovertemplate="Emissor: %{y} — %{customdata[0]}<br>Destino: %{x} — %{customdata[1]}"
                  "<br>Participação: %{z:.2f}%<br>Emissões: %{customdata[2]:,.2f} Gg<extra></extra>",
))
fig_dependencia.update_layout(title="Rede de atribuição: dez maiores emissores → destinos", height=600,
                              xaxis_title="Atividade da demanda final", yaxis_title="Atividade emissora")
fig_dependencia.update_xaxes(type="category", tickangle=90)
fig_dependencia.update_yaxes(type="category", autorange="reversed")
for figura in [fig_concentracao, fig_alcance, fig_dependencia]:
    figura.show()


## 6. Resultados exportados e limites da conclusão

A resposta deve combinar concentração, alcance relevante e participação por destino. Não basta dizer que existe uma ligação: o fechamento de Leontief torna a rede quase completa. Não interpretamos PageRank, caminhos mínimos ou clusters como comprovação de dependência tecnológica.

O efeito proporcional de reduzir a intensidade de um setor mantendo x, L e y fixos é uma identidade contábil do modelo. Não é uma previsão de fechamento do setor, substituição de insumos ou viabilidade de uma política. Esses cenários exigiriam uma análise própria.

In [7]:
from html import escape

for nome, tabela in [("metricas", ranking), ("concentracao", concentracao), ("alcance_grupos", alcance_grupos),
                     ("dependencias_grupos", dependencias_grupos), ("participacao_total_R", R),
                     ("participacao_outras_origens_D", D)]:
    tabela.to_csv(pasta / f"{nome}.csv", index=nome in ["metricas", "participacao_total_R", "participacao_outras_origens_D"],
                  encoding="utf-8", na_rep="NaN")
arestas = pd.DataFrame([{"emissor": i, "destino": j, "participacao": d["weight"], "emissoes_gg": d["emissoes_gg"]}
                       for i,j,d in G.edges(data=True)], columns=["emissor", "destino", "participacao", "emissoes_gg"])
arestas.to_csv(pasta / "arestas_R_5pct.csv", index=False)

visao = ranking.head(10)[["descricao", "participacao_emissoes", "fracao_outros_destinos", "alcance_total_1pct",
                          "alcance_total_5pct", "alcance_total_10pct", "compradores_diretos_A_1pct"]].copy()
visao["participacao_emissoes"] *= 100
visao["fracao_outros_destinos"] *= 100
visao = visao.rename(columns={"participacao_emissoes": "% emissões", "fracao_outros_destinos": "% para outros destinos",
                              "alcance_total_1pct": "destinos ≥1%", "alcance_total_5pct": "destinos ≥5%",
                              "alcance_total_10pct": "destinos ≥10%", "compradores_diretos_A_1pct": "compradores A≥1%"})
resumo5 = alcance_grupos[(alcance_grupos.top_k==5) & (alcance_grupos.limiar==0.10)].iloc[0]
partes = ["<!doctype html><html lang='pt-BR'><meta charset='utf-8'><title>Concentração e alcance dos emissores</title>",
          "<style>body{font-family:Arial,sans-serif;color:#172033;max-width:1300px;margin:30px auto;padding:0 20px;line-height:1.5}"
          "table{border-collapse:collapse;font-size:13px}td,th{padding:8px;border:1px solid #ddd}"
          ".nota{background:#eef5ff;padding:18px}small{overflow-wrap:anywhere}</style>",
          "<h1>CO₂ concentrado e dependência das demais cadeias</h1>",
          f"<p class='nota'>Os cinco maiores emissores concentram <strong>{participacoes.head(5).sum():.1%}</strong>; "
          f"os dez maiores, <strong>{participacoes.head(10).sum():.1%}</strong>. "
          f"São necessários <strong>{n50} setores para 50%</strong> e <strong>{n80} para 80%</strong>. "
          "Há concentração, mas não uma situação em que pouquíssimos setores respondem por quase tudo.</p>",
          f"<p>A rede sem limiar tem {densidade_positiva:.1%} das ligações possíveis. "
          "Contar conexões positivas quase não distingue setores. "
          f"Juntos, os cinco maiores representam ao menos 10% da pegada brasileira de "
          f"<strong>{int(resumo5.destinos_acima_limiar)} dos {int(resumo5.destinos_fora_grupo)} destinos fora do grupo</strong>.</p>",
          "<p>Isso é dependência de origens emissoras na cadeia, não prova de insubstituibilidade de fornecedores. "
          "P cobre emissões brasileiras para demanda final doméstica e exportações. "
          "Os valores usam as intensidades estimadas e hipóteses monetárias do projeto.</p>",
          f"<small>P SHA-256: {sha256((Path('outputs')/'matriz_emissoes_producao_2015.csv').read_bytes()).hexdigest()}</small>",
          "<h2>Os maiores emissores têm o mesmo alcance?</h2>", visao.to_html(float_format=lambda x:f"{x:.2f}"),
          "<p>Destinos ≥5%: quantidade de outras demandas finais em que o emissor origina ao menos 5% das emissões brasileiras "
          "da cadeia, incluindo a emissão própria do destino no denominador. Compradores A≥1%: medida distinta, de compras "
          "diretas nacionais por valor de produção. O próprio setor é excluído das duas contagens.</p>"]
for i, figura in enumerate([fig_concentracao, fig_alcance, fig_dependencia]):
    partes.append(figura.to_html(full_html=False, include_plotlyjs=(i==0)))
partes += ["<h2>Alcance coletivo e sensibilidade aos limiares</h2>",
           alcance_grupos.to_html(index=False,float_format=lambda x:f"{x:.3f}"),
           "<h2>Reprodução e interpretação</h2><p>Execute analise_concentracao_dependencia.ipynb. "
           "Os CSVs preservam métricas dos 67 setores, duas normalizações da rede, alcance coletivo e arestas R≥5%. "
           "As entradas são verificadas por SHA-256; P é confrontada com γx. "
           "As métricas são descritivas e os limiares não são critérios universais de criticidade. "
           "A não inclui importações. Os resultados não simulam a retirada de setores.</p></html>"]
(pasta / "concentracao_dependencia.html").write_text("\n".join(partes),encoding="utf-8")
print("Relatório:", pasta / "concentracao_dependencia.html")


Relatório: outputs\concentracao_dependencia\concentracao_dependencia.html
